# Clear to build process
## Toma varios archivos de inputs en excel para generar el analisis de lanzamiento prellenado, permite agregar datos al mismo y a partir de este y otros inputs genera el archivo CTB.xlsx con informacion necesaria para el analisis de la definicion de lo que se planea lanzar
- V1. 2024-10-21. Version inicial 
- V2. 2024-10-23. Mejoras y correcciones:
    - Se definio el nombre de hoja a leer en el archivo de BOM
    - Se cambio el archivo Manifest por Pendiente
    - Se corrigieron f' por f" para correr en jupyterlab
    - Se agrego formato a celdas del CTB
    - Se agrego el proceso de Propuesta de Lanzamiento
    - Se agrego pend, recibo a la formula de OH Dispo (Validar)
    - Se pone 0 en lugar de valores nulos en CTB
    - Genera sugerencia de lanzamientos
- V3. 2024-11-07
    - Carpeta adicional para las Work Orders
    - Se agrega el BOM Details como input para la columna Primary Stock
- V4. 2024-11-18
    - Se elimina el proceso de Recibos, se tomara solo el archivo Pendiente de Recibos que generara el usuario
    - En el analisis de Lanzamiento se toma en cuenta solo lo del Area BOX
    - Se elimina el ultimo guion en las familias
    - Se agrega la cantidad por bom de cada familia y la cantidad mayor en un bom por componente (QTY PER MAX GRAL)
    - Se agrega la sumatoria de cada familia
    - Se agrega la cantidad en el Forecast de cada componente
    - Se agregan las columnas de Simulacion y CTB
    

# 1. Seleccionar archivos de trabajo
Ejecutar la celda siguiente y seleccione la carpeta y los archivos a procesar

In [ ]:
# Seleccion de archivos de trabajo
import pandas as pd
import os
import ipywidgets as widgets
from IPython.display import display, Markdown
import pickle
from ipyfilechooser import FileChooser
from IPython.display import display
from datetime import datetime, timedelta
from openpyxl.cell import MergedCell
from openpyxl import load_workbook, Workbook
from openpyxl.styles import PatternFill, Border, Side, Alignment, Protection, Font
from openpyxl.formatting.rule import CellIsRule
from tqdm.notebook import tqdm

import warnings
from copy import copy 
import gc

warnings.filterwarnings("ignore")

# Function to show a pop-up message
def show_popup_message(message):
    display(Markdown(f"### **{message}**"))
    

# Decorator to handle the permission error
def handle_permission_error_with_popup(func):
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except PermissionError as e:
            if e.errno == 13:  # Permission denied error
                show_popup_message(f"Error: {e}\nFavor de cerrar el archivo.")
    return wrapper

# Decorate the function where the file is being saved
@handle_permission_error_with_popup
def save_df(df, filepath,sheet_name,index):
    df.to_excel(filepath,sheet_name=sheet_name,index=index)

@handle_permission_error_with_popup
def save_wb(wb, filepath):
    wb.save(filepath)

@handle_permission_error_with_popup
def append_sheet(df,path,sheet_name,index):
    with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=index) 

##### Excel management functions
def find_cell_by_text(ws, text):
    """
    Find the cell containing the specified date in the worksheet.
    
    :param ws: The worksheet object from openpyxl
    :param date_value: The date value to search for (datetime object or string)
    :return: The cell address (e.g., 'B2') or None if not found
    """
    # for row in ws.iter_rows():
    for row in ws[ws.calculate_dimension()]:
        for cell in row:
            if cell.value == text:
                return cell.coordinate  # Return cell address (e.g., 'B2')
    return None 

def find_all_cells_by_text(ws, text):
    """
    Find the cell containing the specified date in the worksheet.
    
    :param ws: The worksheet object from openpyxl
    :param date_value: The date value to search for (datetime object or string)
    :return: The cell address (e.g., 'B2') or None if not found
    """
    # for row in ws.iter_rows():
    cells=[]
    for row in ws[ws.calculate_dimension()]:
        for cell in row:
            if cell.value == text:
                cells.append(cell.coordinate)
    return cells 

def set_number_format(ws,col_name,format):
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    col_head_address=find_cell_by_text(ws,col_name)
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].number_format = format

def fill_column(ws,col_name,fill):
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    col_head_address=ws[find_cell_by_text(ws,col_name)].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].fill=fill

def font_column(ws,col_name,font):
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    col_head_address=ws[find_cell_by_text(ws,col_name)].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].font=font    



def fill_formula(ws,col_name,formula):
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    col_head_address=ws[find_cell_by_text(ws,col_name)].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].value=formula


# Cell colors
light_green='99FF99'
avocato_green='E2EFDA'
dark_blue='002060'
light_blue='DDEBF7'
grey='595959'
dark_red='9C0006'
melon='FCE4D6'
white='FFFFFF'
black='000000'
light_pink='FFC7CE'
light_yellow='FFF2CC'

def is_file_open(filepath):
    # Check if the file exists
    if not os.path.exists(filepath):
        return False  # File does not exist, so treat it as "open" for your logic

    try:
        # Try to open the file in write mode
        with open(filepath, 'a'):
            pass
        return False  # File is not open (no exception raised)
    except PermissionError:
        return True 
    
def get_path(file_selectors,selector_name):
    file_selector=[file_selector for file_selector in file_selectors if file_selector.description==selector_name]
    if len(file_selector)==0:
        show_popup_message(f"No se encontro el selector")
        raise SystemExit()
    file_selector=file_selector[0]
    if not file_selector.value:
        return None
    selected_path=(os.path.join(folder_input.selected,file_selector.value))
    return selected_path

# Save and load state using pickle
def save_state_pickle(state, filename='folder_state.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(state, f)

def load_state_pickle(filename='folder_state.pkl'):
    try:
        with open(filename, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        return {"folder_input": None, "folder_output": None, "selections": {}}

def list_files(folder, extension=".xlsx"):
    """List files in a given folder with a specific extension."""
    return [file for file in os.listdir(folder) if file.lower().endswith(extension)]

def update_selector_files(folder, selectors, filters, state_key):
    """Update file selectors based on filters and selected folder."""
    files_list = list_files(folder)
    for selector, filter_name in zip(selectors, filters):
        filtered_files = [file for file in files_list if filter_name.lower() in file.lower()]
        selector.options = filtered_files
        # Restore previous selections from state if available and valid
        valid_selection = state["selections"].get(filter_name)
        if valid_selection in filtered_files:
            selector.value = valid_selection

def update_file_lists():
    # Update based on general folder input
    if folder_input.selected:
        state["folder_input"] = folder_input.selected
        update_selector_files(folder_input.selected, file_selectors, filters, "selections")
    
    # Update based on specific folder without standards
    if folder_wos.selected:
        state["folder_wos"] = folder_wos.selected
        update_selector_files(folder_wos.selected, [selector for selector in file_selectors if selector.description == 'WOS'], ['WOS'], "selections")

    # Update output and WOS folder states
    if folder_output.selected:
        state["folder_output"] = folder_output.selected

    # Save the updated state
    save_state_pickle(state)

# Function to save the current selections in pickle
def save_selections():
    state["selections"] = {filter_name: selector.value if selector.value else state["selections"].get(filter_name, None) for filter_name, selector in zip(filters, file_selectors)}
    save_state_pickle(state)
     
# Agregar las columnas criticas por archivo
mandatory_cols={'Pendiente':[
                            'Tracking No',
                            'Part No',
                            'Qty',
                            'PO',
                            'Tally No',
                            'Date Shipped'
                            ],
                'Recibos':[
                                'Part No.',
                                'Qty. of TRX',
                                'Reference'
                            ],
                'WOS':[
                    'PO cliente',
                    'Modelo',
                    'WO\n QTY',
                    'WO'
                ],
                'Independent Demands':[
                        'Reference Notes',
                        'Part Name',
                        'P Family',
                        'P.O. Unit Price',
                        'Qty'
                    ],
                'BOMS':[
                    'BOM',
                    'Component',
                    'Qty Per'
                ],
                'BOM Detail':[
                    'Flat Component',
                    'Primary Stock'
                ],                
                'Consumption':[
                    'Site',
                    'Name',
                    'Description',
                    'Note',
                    'Std Unit Cost',
                    'Total',
                    ' PP',
                    'NR',
                    'Allocation',
                    'IssueToWo',
                    'ABC Code',
                    'Buyer Code',
                    'Manufacturer',
                    'MPN',
                    'LT',
                    'UOM',
                    'WhereUsed'
                ],
                'On Hand Detail':[
                    'Part',
                    'Quantity'
                ],
                'Alternos':[
                    'HL1Z (New LED)',
                    'CCT/CRI',
                    'APC (ZES Old LED)'
                ]

                }


def check_mandatory_cols(cols,selector_name):
    missing_columns = [col for col in mandatory_cols[selector_name] if col not in cols]
    if len(missing_columns)>0:
        show_popup_message(f"No se encontraron las siguientes columnas en el archivo {selector_name}: {missing_columns}")
        raise SystemExit()
    return
state = load_state_pickle()

# Create FileChooser for folder input
folder_input = FileChooser(show_only_dirs=True, title="Carpeta de trabajo")
folder_output = FileChooser(show_only_dirs=True, title="Carpeta de salida")
folder_wos = FileChooser(show_only_dirs=True, title="Carpeta de Work Orders")

# Set initial paths if available in the saved state
if state.get("folder_input"):
    if os.path.exists(state["folder_input"]):
        folder_input.reset(path=state["folder_input"])
        folder_input._apply_selection()

if state.get("folder_output"):
    if os.path.exists(state["folder_output"]):
        folder_output.reset(path=state["folder_output"])
        folder_output._apply_selection()        

if state.get("folder_wos"):
    if os.path.exists(state["folder_wos"]):
        folder_wos.reset(path=state["folder_wos"])
        folder_wos._apply_selection()        

# Create 8 Select widgets to filter files by different text
filters = ['Consumption', 'Independent Demands', 'On Hand Detail', 'Pendiente', 'WOS', 'BOMS','BOM Detail','Alternos']
file_selectors = [widgets.Select(options=[], description=filter_name) for filter_name in filters]

# Watch for folder selection changes and update file lists
folder_input.register_callback(lambda x: update_file_lists())
folder_output.register_callback(lambda x: update_file_lists())
folder_wos.register_callback(lambda x: update_file_lists())


# Add listeners to save selections when they change
for selector in file_selectors:
    selector.observe(lambda change: save_selections(), names='value')
    
# Layout: FileChooser on the left, and the Select widgets in two columns
left_column1 = widgets.VBox([folder_input])
left_column2 = widgets.VBox([folder_output])
left_column3 = widgets.VBox([folder_wos])
right_column_1 = widgets.VBox(file_selectors[:4])  # First 4 selectors in the first column
right_column_2 = widgets.VBox(file_selectors[4:])  # Last 4 selectors in the second column


# Display the two-column layout in a single row
ui = widgets.HBox([widgets.VBox([left_column1, left_column2, left_column3]), widgets.HBox([right_column_1, right_column_2])])
display(ui)

# Initialize the file lists with the loaded state
update_file_lists()
selected_files = []
for i, file_selector in enumerate(file_selectors):
    if file_selector.value:  # Check if a file is selected
        selected_files.append(f"{file_selector.description}: {file_selector.value}")
files_list = "\n".join(f"- {file}" for file in selected_files)
# show_popup_message(f"### Selected files:\n{files_list}")


# 2. Analisis de lanzamiento
- Utiliza los siguientes inputs para el analisis de requerimiento por lanzar
    - Independent Demands
    - WOS Lanzadas
- Agregue comentarios en las columnas "Comentarios" y "Status de Lineas"
- El proceso obtendra los comentarios del ultimo analisis de lanzamiento generado por lo que se puede correr varias veces
- No conserva comentarios de registros eliminados, es decir, si la demanda no existe, no apareceran sus comentarios

In [40]:
#Analisis de lanzamiento
path_launch=os.path.join(folder_output.selected,'Analisis_lanzamiento.xlsx')
if is_file_open(path_launch):
    show_popup_message(f"Favor de cerrar el archivo {path_launch}")
    raise SystemExit()   

df_notes=pd.DataFrame()
if os.path.exists(path_launch):
    df_notes=pd.read_excel(path_launch,sheet_name='Analisis de lanzamiento')
    df_notes.dropna(subset=['Comentarios', 'Status de Lineas'], how='all', inplace=True)
    df_notes=df_notes[['PO','MODELO','Comentarios','Status de Lineas']].drop_duplicates()

path_lanzadas=get_path(file_selectors,'WOS')

df_lanzadas_wb=pd.read_excel(path_lanzadas,sheet_name=None)

df_lanzadas=pd.DataFrame()
for sheet in df_lanzadas_wb.keys():
    if 'Seguimiento' in sheet:
        df=df_lanzadas_wb[sheet]
        if 'Unnamed' in df.columns[0]:
            df.columns=df.iloc[0]
            df.drop(0,inplace=True)
        check_mandatory_cols(df.columns,selector_name='WOS')
        df_lanzadas=pd.concat([df_lanzadas,df])
df_lanzadas=df_lanzadas[df_lanzadas['Area']=='BOX']
df_lanzadas_raw=df_lanzadas.copy()
df_lanzadas.rename({"PO cliente":"PO",
                  "Modelo":"MODELO"
                  },axis=1,inplace=True)

df_lanzadas
df_lanzadas=df_lanzadas.groupby(['PO','MODELO']).agg({
    'WO':'first',
    'WO\n QTY':['first','sum']

})

df_lanzadas.columns = ['_'.join(col).strip() for col in df_lanzadas.columns.values]
df_lanzadas.rename({"WO_first":"WO",
                  "WO\n QTY_first":"QTY WO",
                  "WO\n QTY_sum":"Total Wos lanzadas"
                  },axis=1,inplace=True)

df_lanzadas.reset_index(inplace=True)

path_demand=get_path(file_selectors,'Independent Demands')
df_demand=pd.read_excel(path_demand,sheet_name='Independent Demands')
check_mandatory_cols(df_demand.columns,'Independent Demands')

df_demand_raw=df_demand.copy()
df_demand.rename({"Reference Notes":"PO",
                  "Part Name":"MODELO",
                  "P.O. Unit Price":"Price",
                  "Qty":"REQ"
                  },axis=1,inplace=True)


df_demand=df_demand.groupby(['P Family','PO','MODELO']).agg({
    'REQ':'sum',
    'Price':'first'
})
df_demand.reset_index(inplace=True)
df_demand['Sales']=df_demand['Price']*df_demand['REQ']
df_demand=df_demand.merge(df_lanzadas,how='left',on=['PO','MODELO'])

df_demand['WO']=df_demand['WO'].fillna(0).astype(int)
#Familia para la demanda por lanzar
idx=df_demand['PO'].str.contains('FC')
df_demand.loc[~idx,'Familia']=df_demand.loc[~idx,'MODELO'].str[:8]

#Familia para el forecast
df_demand.loc[idx,'Familia']='FC-'+df_demand.loc[idx,'P Family']



df_demand['Familia']=df_demand['Familia'].str.rstrip('-')
df_demand=df_demand[['MODELO','PO','P Family','Familia','REQ','WO','QTY WO','Total Wos lanzadas']]
df_demand[['QTY WO','Total Wos lanzadas']]=df_demand[['QTY WO','Total Wos lanzadas']].fillna(0).astype(int)
df_demand['QTY Pend. Lanzar']=df_demand['REQ']-df_demand['Total Wos lanzadas']


if len(df_notes)>0:
    df_demand=df_demand.merge(df_notes,how='left',on=['PO','MODELO'])
else:
    df_demand['Comentarios']=''
    df_demand['Status de Lineas']=''
df_demand=df_demand.sort_values(['MODELO','PO'])
df_demand_launch=df_demand[~df_demand['PO'].str.contains('FC')]
save_df(df=df_demand_launch,filepath=path_launch,sheet_name='Analisis de lanzamiento',index=False)
append_sheet(df_lanzadas_raw, path_launch,'WOS Lanzadas',index=False)
append_sheet(df_demand_raw, path_launch,'Independent Demands',index=False)
try:
    os.startfile(path_launch)
except:
    show_popup_message(f"El Analisis de lanzamiento esta listo.")

# 3. Consumption
- Crea el archivo CTB tomando los siguientes inputs:
    - BOMs de Lanzamiento
    - Consumption
    - Tabla de alternos
    - Calendario. Es importante llenar el calendario para fechas mas alla de la demanda

## 3.1 Consolidacion de BOM y Demanda

In [192]:
#BOM y Demanda

path_bom=get_path(file_selectors,'BOMS')
if not os.path.exists(path_bom):
    show_popup_message(f"No se ha encontrado el archivo {path_bom}")
    raise SystemExit()   

df_bom=pd.read_excel(path_bom,sheet_name='Flat Bill Browser - Cost Roll U')
check_mandatory_cols(df_bom.columns,'BOMS')
df_bom=df_bom[['BOM','Component','Qty Per']]
df_bom.rename({'BOM':'MODELO'},axis=1,inplace=True)

df_bom_demand=df_demand.copy()

df_bom_demand['Comentarios']=df_bom_demand['Comentarios'].fillna('')
df_bom_demand=df_bom_demand[~df_bom_demand['Comentarios'].str.upper().str.contains('LANZAMIENTO COMPLETO')]

df_bom_demand=df_bom_demand[['PO','MODELO','P Family','Familia','QTY Pend. Lanzar']]

df_bom_demand=df_bom_demand.merge(df_bom,how='left',on='MODELO')

df_bom_demand['Qty Per']=df_bom_demand['Qty Per'].fillna(0)
# Llenar Forecast BOM, el modelo es el componente y la cantidad es 1
df_bom_demand.loc[df_bom_demand['PO'].str.contains('FC'),'Component']=df_bom_demand.loc[df_bom_demand['PO'].str.contains('FC'),'MODELO']
df_bom_demand.loc[df_bom_demand['PO'].str.contains('FC'),'Qty Per']=1
df_bom_demand['REQ. Total']=df_bom_demand['Qty Per']*df_bom_demand['QTY Pend. Lanzar']


df_bom_max=df_bom_demand.groupby(['Component']).max()[['Qty Per']]
df_bom_max.reset_index(inplace=True)
df_bom_max.rename({'Qty Per':'QTY PER MAX GRAL'},axis=1,inplace=True)

path_bom_demand=os.path.join(folder_output.selected,'bom_demand.xlsx')
save_df(df_bom_demand,path_bom_demand,'bom_demand',index=False)


pivot_bom_demand = pd.pivot_table(df_bom_demand, 
                       values=['REQ. Total','Qty Per'],  
                       index=['Component'],         
                       columns=['P Family','Familia'],       
                       aggfunc={'REQ. Total': 'sum', 'Qty Per': 'last'},
                       fill_value=0
                        ) 

# Flatten the columns after the pivot
pivot_bom_demand.columns = [' '.join(col).strip() for col in pivot_bom_demand.columns.values]
# Sort columns by name to intercalate 'REQ. Total' and 'Qty Per' for each 'Familia'
sorted_columns = sorted(pivot_bom_demand.columns, key=lambda x: (x.split()[-1], x.split()[0]))
pivot_bom_demand = pivot_bom_demand[sorted_columns]
pivot_bom_demand.reset_index(inplace=True)
# Lista de columnas para sumatorias por categoria de familia, no se suman las columnas Forecast (FC)
dict_req_total={}
for pfamily in df_bom_demand['P Family'].drop_duplicates().to_list():
    cols_to_sum=[x for x in pivot_bom_demand.columns if ((f"REQ. Total {pfamily}" in x)&('FC-' not in x))]
    pivot_bom_demand[pfamily]=pivot_bom_demand[cols_to_sum].sum(axis=1)
fc_qty_cols=[x for x in pivot_bom_demand.columns if (('FC-' in x)&('Qty Per' in x))]
fc_req_cols=[x for x in pivot_bom_demand.columns if (('FC-' in x)&('REQ. Total' in x))]
pivot_bom_demand.drop(fc_qty_cols,axis=1,inplace=True)
# I will use family columns for the format
family_cols=pivot_bom_demand.columns
family_cols=family_cols[1:]
pivot_bom_demand=pivot_bom_demand.merge(df_bom_max,how='left',on='Component')

## 3.2 Consolidar Consumption
Este proceso debe tomar unos segundos, si llega a un minuto, revisar si hay mensajes en excel

In [193]:
#Consumption
def get_mondays(n):
    today = datetime.today()
    
    # Calculate the Monday of the current week
    monday = today - timedelta(days=today.weekday())  # `today.weekday()` gives 0 for Monday, so we subtract it
    
    # Generate the next `n` Mondays including the current week's Monday
    mondays = [monday + timedelta(weeks=i) for i in range(n + 1)]

    return mondays
path_ctb=os.path.join(folder_output.value,'CTB KRS.xlsx')

path_consumption=get_path(file_selectors,'Consumption')

df_cons=pd.read_excel(path_consumption,header=1)
check_mandatory_cols(df_cons.columns,'Consumption')
#Crear el CTB inicial solo con el RL
wb = load_workbook(path_consumption)
save_wb(wb=wb,filepath=path_ctb)
demand_cols=[x for x in df_cons.columns if "Demand" in x]
scd_cols=[x for x in df_cons.columns if "Schd" in x]
general_cols=['Site',
'Name',
'Description',
'Note',
'Std Unit Cost',
'Total',
' PP',
'NR',
'Allocation',
'IssueToWo',
'ABC Code',
'Buyer Code',
'Manufacturer',
'MPN',
'LT',
'UOM',
'WhereUsed']

ctb_cols=[
    'Name',
    'Description',
    'Note',
    'Total',
    ' PP',
    'NR',
    'Allocation',
]

mondays = get_mondays(len(demand_cols)-1)

df_cols=pd.DataFrame(columns=['demand_cols'],data=demand_cols)
df_cols['scd_cols']=scd_cols
df_cols['monday_date']=mondays
df_cols['monday_date']=pd.to_datetime(df_cols['monday_date']).dt.date.astype(str)
df_cols['week']=pd.to_datetime(df_cols['monday_date']).dt.isocalendar().week
df_calendar=pd.read_excel('calendario.xlsx')
df_calendar['monday_date']=df_calendar['monday_date'].astype(str)
df_calendar['demand_year_month']=df_calendar['year'].astype(str)+'-'+df_calendar['closing_month'].astype(str)
df_calendar['scd_year_month']=df_calendar['demand_year_month'].shift(-1)
df_cols=df_cols.merge(df_calendar,on='monday_date',how='left')
df_rl=pd.DataFrame()
for index,row in df_cols.iterrows():
    df=df_cons[general_cols+[row['demand_cols'],row['scd_cols']]]
    df['demand_year_month']=row['demand_year_month']
    df['scd_year_month']=row['scd_year_month']
    df['monday_date']=row['monday_date']
    df['year']=row['year']
    df.rename({row['demand_cols']:'Demand',row['scd_cols']:'SchdRcpt'},axis=1,inplace=True)
    df_rl=pd.concat([df,df_rl])
    

if len(df_rl[df_rl['demand_year_month'].isnull()])>0:
    show_popup_message("Cuidado, se encontraron fechas sin cierre, favor de completar el calendario")



df_d=df_rl[general_cols+['Demand','demand_year_month','monday_date','year']]
df_d['type']='Demand'
df_d.rename({'Demand':'qty','demand_year_month':'year_month'},axis=1,inplace=True)
df_r=df_rl[general_cols+['SchdRcpt','scd_year_month','monday_date','year']]
df_r['type']='SchdRcpt'
df_r.rename({'SchdRcpt':'qty','scd_year_month':'year_month'},axis=1,inplace=True)
df_rl_raw=pd.concat([df_d,df_r])

df_rl_raw[general_cols]=df_rl_raw[general_cols].fillna('')
df_rl[general_cols]=df_rl[general_cols].fillna('')
rl_raw_path=os.path.join(folder_output.selected,'rl_raw.xlsx')
save_df(df=df_rl_raw,filepath=rl_raw_path,sheet_name='RL',index=False)

df_rl_raw.fillna('',inplace=True)

## 3.3 On hand Detail

In [194]:
#On hand detail
if is_file_open(path_ctb):
    show_popup_message(f"Favor de cerrar un archivo {path_ctb}")
    raise SystemExit()  

path_on_hand=get_path(file_selectors,'On Hand Detail')
df_onhand=pd.read_excel(path_on_hand)
check_mandatory_cols(df_onhand.columns,'On Hand Detail')
df_onhand=df_onhand.groupby(['Part']).sum()[['Quantity']]
df_onhand.reset_index(inplace=True)
append_sheet(df_onhand,path_ctb,'On Hand Detail',False)

df_onhand=df_onhand.groupby(['Part']).sum()[['Quantity']]
df_onhand.reset_index(inplace=True)
append_sheet(df_onhand,path_ctb,'On Hand Detail',False)

## 3.4 Recibos

In [195]:
#Recibos
path_pend_recibo=os.path.join(folder_output.selected,'Pendiente de Recibo.xlsx')
if is_file_open(path_pend_recibo):
    show_popup_message(f"Favor de cerrar un archivo {path_pend_recibo}")
    raise SystemExit()

path_manifest=get_path(file_selectors,'Pendiente')

# path_recibo=get_path(file_selectors,'Recibos')

# df_recibo=pd.read_excel(os.path.join(folder_input.selected,path_recibo))
# check_mandatory_cols(df_recibo.columns,'Recibos')
# df_recibo=df_recibo[mandatory_cols['Recibos']]
# df_recibo['Tally No']=df_recibo['Reference'].str.split('!',expand=True)[1]
# df_recibo=df_recibo.groupby(['Tally No','Part No.']).sum()

# df_recibo.reset_index(inplace=True)
# df_recibo.rename({'Part No.':'Part No'},axis=1,inplace=True)

if os.path.exists(path_pend_recibo):
    df_manifest=pd.read_excel(path_pend_recibo)
    df_manifest_new=pd.read_excel(os.path.join(folder_input.selected,path_manifest))
    check_mandatory_cols(df_manifest_new,'Pendiente')
    df_manifest_new=df_manifest_new.merge(df_manifest,how='left',on=['Tally No', 'Part No'],suffixes=('','_y'))
    df_manifest_new=df_manifest_new[df_manifest_new['Tracking No_y'].isnull()]
    df_manifest_new[mandatory_cols['Pendiente']]
    df_manifest=pd.concat([df_manifest,df_manifest_new])
else:
    df_manifest=pd.read_excel(os.path.join(folder_input.selected,path_manifest))
    check_mandatory_cols(df_manifest,'Pendiente')
    df_manifest=df_manifest[mandatory_cols['Pendiente']]
    

# df_manifest=df_manifest.merge(df_recibo,how='left', on=['Tally No', 'Part No'])

# Manifest menos recibos
# df_manifest=df_manifest[df_manifest['Reference'].isnull()]
df_manifest=df_manifest[mandatory_cols['Pendiente']]

save_df(df_manifest,path_pend_recibo,sheet_name='Pendiente de recibo.xlsx',index=False)

## 3.5 CTB
- No se conservan cambios manuales

In [196]:
#CTB
if is_file_open(path_ctb):
    show_popup_message(f"Favor de cerrar un archivo {path_ctb}")
    raise SystemExit()  

df_rl_raw=pd.read_excel(rl_raw_path)
df_rl_raw.fillna('',inplace=True)

df_alloc=df_rl_raw.drop_duplicates(['Name'])[['Name','Allocation']]
df_alloc.rename({'Name':'Alterno','Allocation':'Aloc Aterno'},axis=1,inplace=True)

#Agregar alternos
path_alternos=get_path(file_selectors,'Alternos')
df_altern=pd.read_excel(path_alternos)
check_mandatory_cols(df_altern.columns,'Alternos')
df_altern=df_altern.merge(df_onhand,left_on='APC (ZES Old LED)',right_on='Part')
df_altern.drop(['APC (ZES Old LED)','CCT/CRI'],axis=1,inplace=True)
df_altern.rename({'HL1Z (New LED)':'Name','Part':'Alterno','Quantity':'ON hand'},axis=1,inplace=True)
df_altern=df_altern.merge(df_alloc,how='left',on='Alterno')
df_rl_raw=df_rl_raw.merge(df_altern,how='left',on='Name')

#Agregar pendiente de recibo
pend_recibo=df_manifest.groupby(['Part No']).sum('Qty')
pend_recibo.reset_index(inplace=True)
pend_recibo.rename({'Part No':'Name','Qty':'pend, recibo'},axis=1,inplace=True)
df_rl_raw=df_rl_raw.merge(pend_recibo,how='left',on='Name')

# Formula columns:
formula_cols=['Req. total','Delta','OH Disp']
for col in formula_cols:
    df_rl_raw[col]=''

path_bom_detail=get_path(file_selectors,'BOM Detail')

if not path_bom_detail:
    df_rl_raw['Primary Stock']=''
else:
    df_bomdetail=pd.read_excel(os.path.join(folder_input.selected,path_bom_detail))
    check_mandatory_cols(df_bomdetail.columns,'BOM Detail')
    df_bomdetail.drop('BOM',axis=1,inplace=True)
    df_bomdetail=df_bomdetail.drop_duplicates(['Flat Component'])
    df_rl_raw=df_rl_raw.merge(df_bomdetail,left_on='Name',right_on='Flat Component')
    df_rl_raw.drop('Flat Component',axis=1,inplace=True)

df_rl_raw.fillna('',inplace=True)


df_ctb = pd.pivot_table(df_rl_raw, 
                       values=['qty'],  
                       index=ctb_cols+formula_cols+['Alterno','ON hand','Primary Stock','Aloc Aterno','pend, recibo'],         
                       columns=['year_month','type'],       
                       aggfunc='sum',
                       fill_value=0
                        )  
df_ctb.reset_index(inplace=True)

# Delta comp for columns
cols=df_ctb['qty'].columns
cols=[col[0] for col in cols]
cols=set(cols)

for col in list(cols):
    if not 'Demand' in df_ctb['qty',col].columns:
        df_ctb.insert(df_ctb.columns.get_loc(('qty',col,'SchdRcpt')),('qty',col,'Demand'),0)

for col in list(cols):
    if not 'SchdRcpt' in df_ctb['qty',col].columns:
        df_ctb.insert(df_ctb.columns.get_loc(('qty',col,'Demand'))+1,('qty',col,'SchdRcpt'),0)

for col in list(cols):
    if not 'Delta Comp' in df_ctb['qty',col].columns:
        df_ctb.insert(df_ctb.columns.get_loc(('qty',col,'SchdRcpt'))+1,('qty',col,'Delta Comp'),0)

df_ctb.rename({
    "Name":"Component",
    "Description":"Component Description"
},axis=1,inplace=True)


if len(pivot_bom_demand)>0:
    existing_cols=df_ctb.columns
    if not isinstance(pivot_bom_demand.columns, pd.MultiIndex):
        tuple_columns = [(col, '', '') for col in pivot_bom_demand.columns]
        multiindex_columns = pd.MultiIndex.from_tuples(tuple_columns)
        pivot_bom_demand.columns=multiindex_columns
    pivot_bom_demand['Simulacion']=''
    pivot_bom_demand['CTB']=''
    df_ctb=df_ctb.merge(pivot_bom_demand,how='left',on='Component')
    remaining_columns = [col for col in df_ctb.columns if col not in pivot_bom_demand.columns]
    df_ctb=df_ctb[list(pivot_bom_demand.columns)+list(remaining_columns)]


# Ordeno columnas de Forecast al inicio
family_cols = family_cols.tolist()
for item in reversed(fc_req_cols):
    if item in family_cols:
        family_cols.remove(item)
        family_cols.insert(0, item)
family_cols = pd.Index(family_cols)

ordered_cols=[('Component', '', '')]+\
            [(col, '', '') for col in family_cols]+\
            [
            (           'Req. total','',''),
            (                'Delta','',''),
            (                'QTY PER MAX GRAL','',''),
            (                'CTB','',''),
            (              'Alterno','',''),
            (              'ON hand','',''),
            (        'Primary Stock','',''),
            ('Component Description','',''),
            (                 'Note','',''),
            (              'OH Disp','',''),
            (                'Total','',''),
            (                  ' PP','',''),
            (                   'NR','',''),
            (           'Allocation','',''),
            (          'Aloc Aterno','',''),
            (         'pend, recibo','',''),
            (         'Simulacion','','')]+\
            [col for col in df_ctb.columns if col[0]=='qty']

            
df_ctb=df_ctb[ordered_cols]

modified_columns = [(col[0], col[1], col[2]) if 'qty' in col[0] else ('', '', col[0]) for col in ordered_cols]
multiindex_columns = pd.MultiIndex.from_tuples(modified_columns)
df_ctb.columns=multiindex_columns
df_ctb.fillna(0,inplace=True)
append_sheet(df_ctb,path_ctb,'CTB',index=True)

### 3.4.1 Formato de CTB

In [ ]:
#Format workbook CTB
if is_file_open(path_ctb):
    show_popup_message(f"Favor de cerrar un archivo {path_ctb}")
    raise SystemExit()  

try:
    wb = load_workbook(path_ctb)
    ws = wb['CTB']

    init=find_cell_by_text(ws,'Component')
    #Reposition headers only if not done already
    if ws[init].column!=1:
        merged_cells = list(ws.merged_cells.ranges)  
        for merged_cell in merged_cells:
                ws.unmerge_cells(str(merged_cell))  
        ws.delete_cols(1)
        ws.delete_rows(4)

        for merged_cell in merged_cells:
            min_col, min_row, max_col, max_row = merged_cell.min_col, merged_cell.min_row, merged_cell.max_col, merged_cell.max_row
            if min_col > 1:
                new_range = ws.cell(min_row, min_col - 1).coordinate + ":" + ws.cell(max_row, max_col - 1).coordinate
                ws.merge_cells(new_range)  

    #Fonts and fills
    for row in ws[ws.calculate_dimension()]:
        for cell in row:
            cell.font=Font(size=8,name='Arial')

    demand_start=find_cell_by_text(ws,'Demand')
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    fill = PatternFill(start_color=light_pink, end_color=light_pink, fill_type="solid")
    font = Font(size=8,name='Arial',color=dark_red) 
    ws.conditional_formatting.add(f"{demand_start}:{last_cell}",  
        CellIsRule(operator='lessThan', formula=['0'], stopIfTrue=True, fill=fill, font=font))

    fill = PatternFill(start_color=grey, end_color=grey, fill_type="solid")
    cell=ws[find_cell_by_text(ws,'Component')]
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=white, bold=True)

    cell=ws[find_cell_by_text(ws,'Primary Stock')]
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=white, bold=True) 
    cell.alignment = Alignment(text_rotation=90,wrap_text=True, horizontal='center', vertical='center')  

    cell=ws[find_cell_by_text(ws,'OH Disp')]
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=white, bold=True)   
    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')

    cell=ws[find_cell_by_text(ws,' PP')]
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=white, bold=True)   
    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center') 

    cell=ws[find_cell_by_text(ws,'NR')]
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=white, bold=True)   
    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center') 

    cell=ws[find_cell_by_text(ws,'Allocation')]
    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=white, bold=True)   

    fill = PatternFill(start_color=dark_blue, end_color=dark_blue, fill_type="solid")

    cell=ws[find_cell_by_text(ws,'Component Description')]
    cell.alignment = Alignment(text_rotation=90,wrap_text=True, horizontal='center', vertical='center')
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=white, bold=True)

    cell=ws[find_cell_by_text(ws,'Note')]
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=white, bold=True)
    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')  

    cell=ws[find_cell_by_text(ws,'Req. total')]
    cell.alignment = Alignment(text_rotation=90,wrap_text=True, horizontal='center', vertical='center')
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=white, bold=True)

    cell=ws[find_cell_by_text(ws,'Delta')]
    cell.alignment = Alignment(text_rotation=90,wrap_text=True, horizontal='center', vertical='center')
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=white, bold=True)    


    cell=ws[find_cell_by_text(ws,'pend, recibo')]
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=white, bold=True)  
    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')  

    cell=ws[find_cell_by_text(ws,'Simulacion')]
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=white, bold=True)  
    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')      

    for col in family_cols:
         cell=ws[find_cell_by_text(ws,col)]
         if 'REQ. Total' in col:
            cell.font=Font(size=8,name='Arial',color=black, bold=False)  
            cell.fill = PatternFill(start_color=light_blue, end_color=light_blue, fill_type="solid")
         elif 'Qty Per' in col:
            cell.font=Font(size=8,name='Arial',color=white, bold=True)  
            cell.fill = PatternFill(start_color=grey, end_color=grey, fill_type="solid")
         cell.alignment=Alignment(text_rotation=90,horizontal='center', vertical='center')

    fill = PatternFill(start_color=light_green, end_color=light_green, fill_type="solid")
    cell=ws[find_cell_by_text(ws,'Alterno')]
    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
    cell.fill=fill
    cell=ws[find_cell_by_text(ws,'ON hand')]
    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
    cell.fill=fill
    cell=ws[find_cell_by_text(ws,'Total')]
    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
    cell.fill=fill
    cell=ws[find_cell_by_text(ws,'Aloc Aterno')]
    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
    cell.fill=fill

    cell=ws[find_cell_by_text(ws,'QTY PER MAX GRAL')]
    cell.alignment = Alignment(text_rotation=90,wrap_text=True, horizontal='center', vertical='center')
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=black, bold=False)    

    cell=ws[find_cell_by_text(ws,'CTB')]
    cell.alignment = Alignment(text_rotation=90,wrap_text=True, horizontal='center', vertical='center')
    cell.fill=fill
    cell.font=Font(size=8,name='Arial',color=black, bold=False)  

    fill = PatternFill(start_color=avocato_green, end_color=avocato_green, fill_type="solid")
    cell1=ws[find_cell_by_text(ws,'Demand')]
    for row in ws[f"{cell1.offset(-1,0).coordinate}:{ws[ws.calculate_dimension()][cell1.row-1][-1].coordinate}"]:
        for cell in row:
            cell.fill=fill
            cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')

    set_number_format(ws,'Delta',format='#,##0.00_);[Red](#,##0.00)')
    set_number_format(ws,'CTB',format='#,##0.00_);[Red](#,##0.00)')
    set_number_format(ws,'OH Disp',format='#,##0.00_);[Red](#,##0.00)')
    font=Font(size=8,name='Arial', bold=True) 
    font_column(ws,'Delta',font)
    font_column(ws,'OH Disp',font)

    fill = PatternFill(start_color=light_yellow, end_color=light_yellow, fill_type="solid")
    fill_column(ws,'Total',fill)

    fill = PatternFill(start_color=melon, end_color=melon, fill_type="solid")
    fill_column(ws,'Req. total',fill)
    fill_column(ws,'Delta',fill)
    fill = PatternFill(start_color=light_blue, end_color=light_blue, fill_type="solid")
    fill_column(ws,'OH Disp',fill)
    fill_column(ws,'pend, recibo',fill)

    # Formulas

    # Suma de familias
    init=find_cell_by_text(ws,family_cols[0])
    end=find_cell_by_text(ws,family_cols[-1])
    col_head_address=ws[find_cell_by_text(ws,'Req. total')].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].value=f"=sum({ws.cell(row[0].row,ws[init].column).coordinate}:{ws.cell(row[0].row,ws[end].column).coordinate})"


    # Formula Delta
    cell_a=find_cell_by_text(ws,'OH Disp')
    cell_b=find_cell_by_text(ws,'Req. total')
    col_head_address=ws[find_cell_by_text(ws,'Delta')].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].value=f"={ws.cell(row[0].row,ws[cell_a].column).coordinate}-{ws.cell(row[0].row,ws[cell_b].column).coordinate}"

    # Formula CTB
    cell_a=find_cell_by_text(ws,'OH Disp')
    cell_b=find_cell_by_text(ws,'QTY PER MAX GRAL')
    col_head_address=ws[find_cell_by_text(ws,'CTB')].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].value=f"={ws.cell(row[0].row,ws[cell_a].column).coordinate}/{ws.cell(row[0].row,ws[cell_b].column).coordinate}"


    # Formula OH Dispo
    cell_a=find_cell_by_text(ws,'Total')
    cell_b=find_cell_by_text(ws,'Allocation')
    cell_c=find_cell_by_text(ws,'pend, recibo')
    cell_d=find_cell_by_text(ws,'Simulacion')
    col_head_address=ws[find_cell_by_text(ws,'OH Disp')].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].value=f"={ws.cell(row[0].row,ws[cell_a].column).coordinate}-{ws.cell(row[0].row,ws[cell_b].column).coordinate}+{ws.cell(row[0].row,ws[cell_c].column).coordinate}+{ws.cell(row[0].row,ws[cell_d].column).coordinate}"

    # Formula for Delta Comp first column
    cell_a=find_cell_by_text(ws,'Total')
    cell_b=find_cell_by_text(ws,'Demand')
    cell_c=find_cell_by_text(ws,'SchdRcpt')
    col_head_address=ws[find_cell_by_text(ws,'Delta Comp')].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].value=f"={ws.cell(row[0].row,ws[cell_a].column).coordinate}+{ws.cell(row[0].row,ws[cell_c].column).coordinate}-{ws.cell(row[0].row,ws[cell_b].column).coordinate}"

    # Formula for Delta Comp
    cells_b=find_all_cells_by_text(ws,'Demand')
    cells_c=find_all_cells_by_text(ws,'SchdRcpt')
    cells_d=find_all_cells_by_text(ws,'Delta Comp')
    df=pd.DataFrame({'Demand':cells_b,
                    'SchdRcpt':cells_c,
                    'DeltaComp':cells_d})

    df['PrevDeltaComp']=df['DeltaComp'].shift(1)
    # The first formula is different so we drop the first row
    df.drop(0,inplace=True)
    for index,dfrow in df.iterrows():
        for row in ws[f"{ws[dfrow['DeltaComp']].offset(1,0).coordinate}:{last_cell}"]:
            row[0].value=f"={ws.cell(row[0].row,ws[dfrow['PrevDeltaComp']].column).coordinate}+{ws.cell(row[0].row,ws[dfrow['SchdRcpt']].column).coordinate}-{ws.cell(row[0].row,ws[dfrow['Demand']].column).coordinate}"



    save_wb(wb=wb,filepath=path_ctb)

except Exception as e:
     show_popup_message(f"Error: CTB no fue formateado. {str(e)}, Corra la generacion de CTB de nuevo")
     

try:
    os.startfile(path_ctb)
except:
    show_popup_message(f"El archivo CTB esta listo.")

# Sugerencia de lanzamientos
- Guardar el archivo CTB antes de correr esta seccion, incluso si no se han realizado cambios en el archivo con el fin de que las formulas se evaluen

In [ ]:
# Funciones
path_launch_suggestion=os.path.join(folder_output.selected,'Launch_suggestion.xlsx')
if is_file_open(path_launch_suggestion):
    show_popup_message(f"Favor de cerrar un archivo {path_launch_suggestion}")
    raise SystemExit()

critical_components=['APC-']

def substract_bom(df_demand, df_mat):
    """
    Subtracts the Bill of Materials (BOM) demand from available materials and calculates completion percentages and shortages.

    Parameters:
    df_demand (DataFrame): DataFrame containing demand data with at least 'component' and 'QTY' columns.
    df_mat (DataFrame): DataFrame containing material availability data with at least 'component' and 'AVAIL' columns.

    Returns:
    df_demand_new (DataFrame): Updated demand DataFrame with additional columns 'completion', 'new_avail', 'short'.
    df_mat_new (DataFrame): Updated material availability DataFrame with adjusted 'AVAIL' after subtracting demand.
    """
    df_demand_new=df_demand.copy()
    df_demand_new=df_demand_new[df_demand_new['QTY Pend. Lanzar']>0]
    df_demand_new=df_demand_new[df_demand_new['REQ. Total']>0]
    df_demand_new['QTY Pend. Lanzar']=1
    df_mat_new=df_mat.copy()
    df_mat_new['OH Disp']=df_mat['OH Disp'].clip(lower=0) # Ignoro disponibilidad negativa
    df_demand_new = df_demand_new.merge(df_mat_new[['Component', 'OH Disp']], how='left', on='Component')
    df_demand_new['OH Disp'].fillna(0,inplace=True)
    df_demand_new['completion'] = (df_demand_new['OH Disp'] / df_demand_new['Qty Per']).clip(upper=1)
    df_demand_new['new_avail'] = (df_demand_new['OH Disp'] - df_demand_new['Qty Per']).clip(lower=0)
    df_demand_new['short'] = (df_demand_new['OH Disp'] - df_demand_new['Qty Per']).clip(upper=0)
    df_demand_total = df_demand_new.groupby('Component')['Qty Per'].sum().reset_index()
    df_mat_new = df_mat_new.merge(df_demand_total, how='left', on='Component')
    df_mat_new['Qty Per'] = df_mat_new['Qty Per'].fillna(0)
    df_mat_new['OH Disp'] = (df_mat_new['OH Disp'] - df_mat_new['Qty Per']).clip(lower=0)
    df_mat_new = df_mat_new.drop(columns=['Qty Per'])
    df_demand_new.fillna(0,inplace=True)
    df_demand_new.reset_index(drop=True,inplace=True)
    df_mat_new.fillna(0,inplace=True)
    return df_demand_new, df_mat_new

def get_bom_completion(df_demand,df_mat):
    df_completion,df_mat_new=substract_bom(df_demand=df_demand,df_mat=df_mat)
    # Ignore orders with critical components (Like APC) completiion less than 0 since they cannot be started
    df_to_drop=pd.DataFrame()
    for component in critical_components:
        df=df_completion[(df_completion['Component'].str.contains(component)) & (df_completion['completion']<1)][['PO','MODELO','QTY Pend. Lanzar']]
        df_to_drop=pd.concat([df,df_to_drop])
    df_to_drop=df_to_drop.drop_duplicates()
    df_to_drop['tmp']=df_to_drop['PO']+df_completion['MODELO']
    df_completion['tmp']=df_completion['PO']+df_completion['MODELO']
    df_completion=df_completion[~df_completion['tmp'].isin(df_to_drop['tmp'])]
    df_completion=df_completion.groupby(['PO','MODELO']).mean('completion')[['completion']]
    df_completion.reset_index(inplace=True)
    df_completion=df_completion.sort_values(['completion'],ascending=False)
    return df_completion
# Cargar datos de excel incluyendo modificaciones manuales
wb=load_workbook(path_ctb, data_only=True)
ws=wb['CTB']
data = ws.values
for _ in range(2):  # Skip the first two rows
    next(data)
columns = next(data)
df_ctb = pd.DataFrame(data, columns=columns)
if df_ctb[df_ctb['OH Disp'].isnull()].shape[0]>0:
    show_popup_message(f"Favor de Guardar, luego cerrar el archivo {path_ctb}")
    raise SystemExit()
# Consolidar todo el bom con la demanda para obtener metrico de completion
#Elimino demanda sin pendientes de lanzar
df_bom_demand=pd.read_excel(path_bom_demand)


#Obtiene el promedio de porcentaje de finalizacion si el bom estuviera disponible para cada linea de demanda
df_available=df_ctb[['Component','OH Disp']]
df_bom_demand=df_bom_demand[['PO','MODELO','QTY Pend. Lanzar','Component','Qty Per','REQ. Total']]
df_available_left=df_available.copy()
df_launch=pd.DataFrame()
alloc_priority=0
completion_left=1
max_req=0
old_req=0

while completion_left>0:
    df_bom_demand=df_bom_demand[df_bom_demand['QTY Pend. Lanzar']>0]
    df_completion=get_bom_completion(df_bom_demand,df_available_left)
    completion_left=df_completion.shape[0]
    if completion_left==0:
        break
    if max_req==0:
        # key=df_completion['PO']+df_completion['MODELO']
        # max_req=df_bom_demand[(df_bom_demand['PO']+df_bom_demand['MODELO']).isin(key)][['PO','MODELO','QTY Pend. Lanzar']].drop_duplicates()['QTY Pend. Lanzar'].sum()
        max_req=completion_left
        progress_bar = tqdm(total=completion_left, desc="Processing",bar_format="{desc}: {percentage:3.0f}% |{bar}|")
        progress_bar.update()
        old_req=max_req
    # new_req=df_bom_demand[(df_bom_demand['PO']+df_bom_demand['MODELO']).isin(key)][['PO','MODELO','QTY Pend. Lanzar']].drop_duplicates()['QTY Pend. Lanzar'].sum()
    new_req=completion_left
    inc_req=old_req-new_req
    progress_bar.update(inc_req)
    old_req=new_req
    po_row=df_completion.loc[df_completion['completion'].idxmax()]
    idx=(df_bom_demand['PO']==po_row['PO']) & (df_bom_demand['MODELO']==po_row['MODELO'])
    df,df_available_left=substract_bom(df_bom_demand[idx],df_available_left)
    df['alloc_priority']=alloc_priority
    alloc_priority+=1
    df_launch=pd.concat([df,df_launch])
    df_bom_demand.loc[idx,'QTY Pend. Lanzar']=df_bom_demand.loc[idx,'QTY Pend. Lanzar']-1    
df_launch.sort_values('alloc_priority',inplace=True)


## Guardar sugerencia de lanzamientos

In [16]:

save_df(df_launch,filepath=path_launch_suggestion,sheet_name='Launch Suggestion',index=False)

df_launch_resume=df_launch[['alloc_priority','PO','MODELO','QTY Pend. Lanzar']].drop_duplicates()
df_launch_resume.sort_values('alloc_priority',inplace=True)
df_launch_resume['group'] = (df_launch_resume['PO'] != df_launch_resume['PO'].shift(1)) | (df_launch_resume['MODELO'] != df_launch_resume['MODELO'].shift(1))
df_launch_resume['group'] = df_launch_resume['group'].cumsum()
df_launch_resume=df_launch_resume.groupby(['group','PO','MODELO']).sum('QTY Pend. Lanzar')[['QTY Pend. Lanzar']]
df_launch_resume.reset_index(inplace=True)

append_sheet(df_launch_resume,path_launch_suggestion,sheet_name='Launch Suggestion Resume',index=False)

os.startfile(path_launch_suggestion)